# Anomaly detection with PyOD

Notebook version of [AnomalyDetectionUsingPyOD](https://github.com/JaminJeong/AnomalyDetectionUsingPyOD) (daily minimum temperatures, sliding windows, multiple detectors). Slots that originally required **Numba** or **torch** are filled with **sklearn substitute** detectors (clearly labeled—not the same algorithms as upstream PyOD). Temperatures are loaded with **`%%cribl_search`** using Cribl Search [`externaldata`](https://docs.cribl.io/search/externaldata/) (same public CSV as the upstream example—see [daily-min-temperatures.csv](https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv) and [Machine Learning Mastery](https://machinelearningmastery.com/time-series-datasets-for-machine-learning/)). [PyOD](https://github.com/yzhao062/pyod) is installed in **Setup** via `micropip`.

## How to run

1. Run **Setup** once (downloads PyOD stack; first run can take several minutes).
2. Run **Shared preprocessing** (Python helpers and constants).
3. Run **Load temperatures** (`%%cribl_search` with `externaldata`) — requires this notebook to run **inside a hosted Cribl app** where Search is available (like other `%%cribl_search` examples).
4. Run **Materialize dataframe**, then **train/test windows**, then each **detector** cell (each draws its own figure). Sections titled **sklearn substitute for …** replace PyOD models that need **Numba** or **PyTorch** with scikit-learn (or pure NumPy) alternatives—they are **not** the same algorithms as upstream PyOD. **Variational Auto Encoder** and **Auto Encoder** slots use PCA / MLP reconstruction scores instead of torch. The upstream **LOCI** screenshots are not part of the upstream script's active models; this notebook keeps the same **18** slots as `anomaly_detection_all_model.py`.

### Setup

Installs **SciPy**, **scikit-learn**, and **joblib**, then **PyOD** with `deps=False` (PyPI also declares **Numba**, which has no Pyodide wheel—PyOD models that import Numba are not used here; see **sklearn substitute** sections instead). Requires PyPI/jsDelivr (allow-listed in `proxies.yml`).

In [ ]:
import micropip

# PyPI declares `numba` for pyod, but Numba has no Pyodide wheel — install the
# scientific stack explicitly, then pyod without transitive deps.
# Pin versions to the Pyodide full-distribution wheel set for this app
# (see `PYODIDE_RELEASE` / bundled `pyodide-lock.json`; revisit pins when bumping Pyodide).
await micropip.install([
    'scipy==1.14.1',
    'scikit-learn==1.7.0',
    'joblib==1.4.2',
])
await micropip.install(['pyod==2.0.5'], deps=False)

### Shared preprocessing

Imports, constants (`window_size`, `contamination`), helper functions for sliding windows and plots, and **sklearn adapter classes** for substitute detectors. Also builds the **neighbor grid** used by the **sklearn substitute for LSCP** cell. Run **Setup** above first so `pyod` is available for detectors that still use it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

window_size = 30
contamination = 0.25
outliers_fraction = contamination
random_state = 42


def make_data_sampling(data, window_size):
    n_samples = len(data)
    if n_samples < window_size:
        raise ValueError('window size is too long for series')
    return np.array([data[i : i + window_size] for i in range(n_samples - window_size + 1)])


def plot_anomalies(title, real_value, y_pred):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(np.arange(len(real_value)), real_value, label='real')
    scatter_x = [i for i, lab in enumerate(y_pred) if lab == 1]
    scatter_y = [real_value[i] for i in scatter_x]
    ax.scatter(scatter_x, scatter_y, color='0.5', edgecolor='r', label='anomaly')
    ax.set_title(title)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def run_detector(title, clf):
    clf.fit(df_data_train)
    y_pred = clf.predict(df_data_test)
    real_value = df_data_test[:, -1]
    plot_anomalies(title, real_value, y_pred)


class SklearnNegLabelsToPyOD:
    # Wrap sklearn: predict 1=inlier, -1=outlier → PyOD-style 1=anomaly, 0=normal

    def __init__(self, estimator):
        self._e = estimator

    def fit(self, X):
        self._e.fit(X)
        return self

    def predict(self, X):
        y = self._e.predict(X)
        return np.where(y == -1, 1, 0)


class GMMLogDensityOutlier:
    # Low Gaussian log-density (cluster-flavored CBLOF substitute)

    def __init__(self, contamination, random_state, n_components=15):
        self.contamination = float(contamination)
        self.random_state = random_state
        self.n_components = int(n_components)
        self._gmm = None
        self._threshold = None

    def fit(self, X):
        from sklearn.mixture import GaussianMixture

        n_max = max(1, min(self.n_components, X.shape[0] // 3, X.shape[0] - 1))
        self._gmm = GaussianMixture(
            n_components=n_max,
            covariance_type='full',
            random_state=self.random_state,
            reg_covar=1e-6,
            max_iter=200,
        )
        self._gmm.fit(X)
        logp = self._gmm.score_samples(X)
        self._threshold = np.quantile(logp, self.contamination)
        return self

    def predict(self, X):
        logp = self._gmm.score_samples(X)
        return (logp < self._threshold).astype(int)


class PCAMSEReconstructionOutlier:
    # High PCA reconstruction MSE (linear VAE-style substitute)

    def __init__(self, contamination, n_components=8, random_state=42):
        self.contamination = float(contamination)
        self.n_components = int(n_components)
        self.random_state = random_state
        self._pca = None
        self._thr = None

    def fit(self, X):
        from sklearn.decomposition import PCA

        ncomp = max(1, min(self.n_components, X.shape[1] - 1, max(1, X.shape[0] - 2)))
        self._pca = PCA(n_components=ncomp, random_state=self.random_state)
        self._pca.fit(X)
        z = self._pca.transform(X)
        recon = self._pca.inverse_transform(z)
        err = np.mean((X - recon) ** 2, axis=1)
        self._thr = np.quantile(err, 1.0 - self.contamination)
        return self

    def predict(self, X):
        z = self._pca.transform(X)
        recon = self._pca.inverse_transform(z)
        err = np.mean((X - recon) ** 2, axis=1)
        return (err > self._thr).astype(int)


class MLPIdentityReconstructionOutlier:
    # Small MLP identity reconstruction (autoencoder substitute)

    def __init__(self, contamination, max_iter=120, random_state=42):
        self.contamination = float(contamination)
        self.max_iter = int(max_iter)
        self.random_state = random_state
        self._mlp = None
        self._thr = None

    def fit(self, X):
        from sklearn.neural_network import MLPRegressor

        n_in = X.shape[1]
        h = max(2, min(32, n_in * 2))
        self._mlp = MLPRegressor(
            hidden_layer_sizes=(h, h),
            activation='relu',
            max_iter=self.max_iter,
            random_state=self.random_state,
            early_stopping=True,
            validation_fraction=0.12,
        )
        self._mlp.fit(X, X)
        recon = self._mlp.predict(X)
        err = np.mean((X - recon) ** 2, axis=1)
        self._thr = np.quantile(err, 1.0 - self.contamination)
        return self

    def predict(self, X):
        recon = self._mlp.predict(X)
        err = np.mean((X - recon) ** 2, axis=1)
        return (err > self._thr).astype(int)


class LOFMajorityVoteOutlier:
    # Majority vote over novelty LOF models (LSCP-style substitute)

    def __init__(self, neighbor_values, contamination):
        from sklearn.neighbors import LocalOutlierFactor

        self._estimators = []
        for n in neighbor_values:
            nn = int(max(2, int(n)))
            self._estimators.append(
                LocalOutlierFactor(n_neighbors=nn, novelty=True, contamination=contamination)
            )

    def fit(self, X):
        for e in self._estimators:
            e.fit(X)
        return self

    def predict(self, X):
        preds = np.stack([e.predict(X) for e in self._estimators], axis=0)
        votes_out = np.sum(preds == -1, axis=0)
        need = preds.shape[0] // 2 + 1
        return (votes_out >= need).astype(int)


### Load temperatures (`externaldata`)

Fetches the CSV through **[`externaldata`](https://docs.cribl.io/search/externaldata/)** so retrieval happens in **Cribl Search**, not via `pd.read_csv` in Pyodide. Uses stock datatype **`CSV Datatypes`**. If Search rejects the datatype on your tenant, try **`CSV`** (v1 stock) or your admin's comma-separated datatype.

In [ ]:
%%cribl_search var=temp_df lang=kql limit=0 preview=false earliest=-50y latest=now
externaldata
[
  "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
import io

import pandas as pd

# externaldata + generic_csv often lands each CSV row in one Search field (e.g. `Event` / `_raw`),
# not as separate Date/Temp columns — rebuild the CSV text then parse.
_cols_lower = {str(c).strip().lower(): c for c in temp_df.columns}
_raw_col = None
for key in ('event', '_raw'):
    if key in _cols_lower:
        _raw_col = _cols_lower[key]
        break
if _raw_col is None:
    for c in temp_df.columns:
        if str(c).startswith('_'):
            _raw_col = c
            break
if _raw_col is None:
    non_time = [c for c in temp_df.columns if str(c).lower() not in ('time', '_time')]
    _raw_col = non_time[-1] if non_time else temp_df.columns[-1]

_lines = temp_df[_raw_col].astype(str).str.strip()
_lines = _lines[_lines.str.len() > 0]
# Drop repeated header rows (some tenants echo the CSV header per event)
_hdr = '"Date","Temp"'
_lines = _lines[_lines != _hdr]
_csv_text = '\n'.join([_hdr] + _lines.tolist())
df_data = pd.read_csv(io.StringIO(_csv_text))
df_data.columns = [str(c).strip() for c in df_data.columns]
if 'Temp' not in df_data.columns:
    raise ValueError(f'Expected Temp column after CSV parse; got {list(df_data.columns)}')
df_data['Temp'] = pd.to_numeric(df_data['Temp'], errors='coerce')
df_data = df_data.dropna(subset=['Temp']).reset_index(drop=True)
df_data.head()

In [ ]:
raw_train, raw_test = train_test_split(
    np.array(df_data['Temp']), test_size=0.2, shuffle=False
)

n_train_windows = len(raw_train) - window_size + 1
if n_train_windows < 2:
    raise ValueError(f'Not enough training points for window_size={window_size}; got {len(raw_train)} temps')

# sklearn / pyod require n_neighbors < n_samples (training rows are sliding windows)
_nn_cap = max(1, n_train_windows - 1)


def _cap_nn(k: int) -> int:
    return max(1, min(int(k), _nn_cap))


LOF_N = _cap_nn(35)
COF_N = LOF_N

_nn_grid = (5, 10, 15, 20, 25, 30, 35, 40, 45, 50)
lscp_neighbor_values = []
_seen = set()
for k in _nn_grid:
    v = _cap_nn(k)
    if v not in _seen:
        _seen.add(v)
        lscp_neighbor_values.append(v)
if len(lscp_neighbor_values) < 2:
    lscp_neighbor_values = [_nn_cap, max(1, _nn_cap // 2)]

df_data_train = make_data_sampling(raw_train, window_size)
df_data_test = make_data_sampling(raw_test, window_size)
print(
    'train windows',
    df_data_train.shape,
    'test windows',
    df_data_test.shape,
    'LOF_N',
    LOF_N,
    'LSCP sklearn neighbor grid',
    lscp_neighbor_values,
)


### Train series (subsampled)

Same idea as `GenGraph('train')` in the upstream `save_graph.py`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(raw_train[::30])
ax.set_title('Train temperatures (every 30th point)')
plt.tight_layout()
plt.show()
plt.close(fig)

### Test series (subsampled)

Same idea as `GenGraph('test')`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(raw_test[::30])
ax.set_title('Test temperatures (every 30th point)')
plt.tight_layout()
plt.show()
plt.close(fig)

## Detectors

Each section fits **one** model on `df_data_train`, scores `df_data_test`, and plots the **last timestep** of each window against anomalies (matching `BasicGenGraph` in the upstream project). Headings **sklearn substitute for …** use scikit-learn (or a small sklearn MLP) instead of the original PyOD model when that model would require **Numba** or **PyTorch** in the browser. Under each heading, a short paragraph explains what the upstream method does and what this notebook runs instead (or which PyOD class is used).

### sklearn substitute for Angle-based Outlier Detector (ABOD)

**Original idea (PyOD):** ABOD scores points using angular relationships between vectors in feature space, so outliers are directions that disagree with most of the data. **This cell:** we use a **nonlinear One-Class SVM with an RBF kernel**, which learns a smooth boundary around the bulk of training windows instead of angular scores. It is a standard, WASM-friendly density-envelope method—not the same math as ABOD, but it targets the same goal: windows whose pattern of temperatures is unlike the majority.

In [ ]:
title = 'sklearn substitute for Angle-based Outlier Detector (ABOD) (RBF One-Class SVM)'
from sklearn.svm import OneClassSVM

clf = SklearnNegLabelsToPyOD(OneClassSVM(kernel='rbf', gamma='scale', nu=outliers_fraction))
run_detector(title, clf)


### sklearn substitute for Cluster-based Local Outlier Factor (CBLOF)

**Original idea (PyOD):** CBLOF clusters the data, then compares each point to its cluster context so dense clusters and sparse clusters are treated fairly for local outliers. **This cell:** we fit a **Gaussian mixture** on training windows and treat **low mixture log-density** (under the learned components) as anomalous—cluster structure plus a density score, without PyOD’s cluster-wise LOF machinery or Numba.

In [ ]:
title = 'sklearn substitute for Cluster-based Local Outlier Factor (CBLOF) (Gaussian mixture log-density)'
clf = GMMLogDensityOutlier(outliers_fraction, random_state, n_components=15)
run_detector(title, clf)


### sklearn substitute for Feature Bagging

**Original idea (PyOD):** Feature Bagging trains base detectors on random subsets of features and combines their opinions so no single feature dimension dominates. **This cell:** we run **PCA** to a fixed-size linear subspace, then **novelty Local Outlier Factor** on the reduced vectors—reduction plus local density in one pipeline. It is an interpretable stand-in for “subspace + ensemble of local views,” not a literal feature-bagging implementation.

In [ ]:
title = 'sklearn substitute for Feature Bagging (PCA + novelty LOF on reduced space)'
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline

n_pca = max(2, min(10, df_data_train.shape[1] - 1))
nn_fb = max(2, min(LOF_N, df_data_train.shape[0] - 1))
pipe = Pipeline(
    [
        ('pca', PCA(n_components=n_pca, random_state=random_state)),
        (
            'lof',
            LocalOutlierFactor(n_neighbors=nn_fb, novelty=True, contamination=outliers_fraction),
        ),
    ]
)
clf = SklearnNegLabelsToPyOD(pipe)
run_detector(title, clf)


### sklearn substitute for Histogram-base Outlier Detection (HBOS)

**Original idea (PyOD):** HBOS builds histograms along features and flags points that land in rare bins—fast, axis-aligned density sketches. **This cell:** we use **scikit-learn’s Isolation Forest**, which isolates points using random axis-aligned splits; trees indirectly approximate multivariate density, so the story is “fast tree-based rarity” rather than explicit histogram bins, but it runs entirely on wheels Pyodide already supports.

In [ ]:
title = 'sklearn substitute for Histogram-base Outlier Detection (HBOS) (IsolationForest)'
from sklearn.ensemble import IsolationForest

clf = SklearnNegLabelsToPyOD(
    IsolationForest(contamination=outliers_fraction, random_state=random_state, n_estimators=200)
)
run_detector(title, clf)


### Isolation Forest

**Algorithm:** Isolation Forest randomly partitions feature space with trees; points that require few splits to isolate are treated as anomalies because they sit in sparse regions of the data. **This cell:** uses **PyOD’s `IForest`** on sliding windows of temperatures, matching the upstream notebook’s tree-based baseline while staying compatible with the pinned scientific stack in Pyodide.

In [ ]:
title = 'Isolation Forest'
try:
    from pyod.models.iforest import IForest
    clf = IForest(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### K Nearest Neighbors (KNN)

**Algorithm:** Classic distance-based kNN outlier detection compares each point to its *k*-nearest neighbors in feature space; isolated or distant windows get high scores. **This cell:** **PyOD `KNN`** with default aggregation—good when local neighborhoods of the multivariate window vectors carry the signal for cold snaps or unusual short patterns.

In [ ]:
title = 'K Nearest Neighbors (KNN)'
try:
    from pyod.models.knn import KNN
    clf = KNN(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Average KNN

**Algorithm:** Instead of a single neighbor distance, average KNN smooths the score by averaging distances to the *k* neighbors, reducing noise from one unlucky neighbor. **This cell:** **PyOD `KNN` with `method='mean'`**—slightly more stable local density view for the same window features.

In [ ]:
title = 'Average KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='mean', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Median KNN

**Algorithm:** Median KNN uses the median of neighbor distances, which is robust if a few neighbors are themselves unusual but the bulk of the neighborhood is still representative. **This cell:** **PyOD `KNN` with `method='median'`**—a robust local-distance variant on the temperature windows.

In [ ]:
title = 'Median KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='median', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Local Outlier Factor (LOF)

**Algorithm:** LOF compares each point’s local reachability density to that of its neighbors; points in sparser pockets than their peers are outliers. **This cell:** **PyOD `LOF`** with **`LOF_N`** neighbors capped from the training window count so `n_neighbors` stays valid—local contrast on the same sliding-window representation as the rest of the notebook.

In [ ]:
title = 'Local Outlier Factor (LOF)'
try:
    from pyod.models.lof import LOF
    clf = LOF(n_neighbors=LOF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Minimum Covariance Determinant (MCD)

**Algorithm:** MCD estimates a robust covariance ellipsoid that ignores a fraction of extreme training points, then flags test points far from that bulk in Mahalanobis distance—effective when “normal” windows are roughly elliptical in feature space. **This cell:** **PyOD `MCD`** with the notebook’s contamination setting.

In [ ]:
title = 'Minimum Covariance Determinant (MCD)'
try:
    from pyod.models.mcd import MCD
    clf = MCD(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### One-class SVM (OCSVM)

**Algorithm:** One-class SVM finds a kernelized boundary around the training bulk; points outside the margin are anomalies—useful when normal behavior is not axis-aligned or Gaussian. **This cell:** **PyOD `OCSVM`** on window vectors, distinct from the earlier **sklearn** RBF One-Class SVM used as an ABOD substitute (different implementation path through PyOD).

In [ ]:
title = 'One-class SVM (OCSVM)'
try:
    from pyod.models.ocsvm import OCSVM
    clf = OCSVM(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Principal Component Analysis (PCA)

**Algorithm:** PCA-based outlier detection uses deviation from the main variance subspace—either reconstruction error or feature-bag norms—so windows that do not align with dominant co-movement of temperatures stand out. **This cell:** **PyOD’s PCA detector** with fixed contamination, complementing the separate linear PCA reconstruction substitute used for the VAE slot.

In [ ]:
title = 'Principal Component Analysis (PCA)'
try:
    from pyod.models.pca import PCA
    clf = PCA(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### sklearn substitute for Stochastic Outlier Selection (SOS)

**Original idea (PyOD):** SOS builds a stochastic neighborhood model and propagates outlierness through that graph. **This cell:** we use **SGDOneClassSVM**, a linear one-class boundary trained with a stochastic optimizer. The “stochastic” flavor is in the optimization path, not SOS’s graph; it gives a simple linear separator between typical and atypical windows when a margin is a reasonable approximation.

In [ ]:
title = 'sklearn substitute for Stochastic Outlier Selection (SOS) (SGDOneClassSVM)'
from sklearn.linear_model import SGDOneClassSVM

clf = SklearnNegLabelsToPyOD(SGDOneClassSVM(nu=outliers_fraction, random_state=random_state))
run_detector(title, clf)


### sklearn substitute for Locally Selective Combination (LSCP)

**Original idea (PyOD):** LSCP combines several base detectors and focuses on regions where local models disagree or are most selective. **This cell:** we train several **novelty LocalOutlierFactor** models with **different neighbor counts** on the same training windows and **majority-vote** their outlier labels on the test set—combination of local density estimates without importing PyOD’s LSCP (which pulls Numba through shared utilities).

In [ ]:
title = 'sklearn substitute for Locally Selective Combination (LSCP) (majority novelty LOF)'
clf = LOFMajorityVoteOutlier(lscp_neighbor_values, outliers_fraction)
run_detector(title, clf)


### Connectivity-Based Outlier Factor (COF)

**Algorithm:** COF measures how “connected” a point is along paths through its neighbors, emphasizing spatial chaining of densities rather than raw *k*-NN distance alone—helpful when outliers sit in structured low-density filaments. **This cell:** **PyOD `COF`** with **`COF_N`** neighbors aligned to the same cap as LOF so training geometry stays valid.

In [ ]:
title = 'Connectivity-Based Outlier Factor (COF)'
try:
    from pyod.models.cof import COF
    clf = COF(n_neighbors=COF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### sklearn substitute for Subspace Outlier Detection (SOD)

**Original idea (PyOD):** SOD searches multiple lower-dimensional projections where a point appears extreme. **This cell:** we apply **TruncatedSVD** to map windows into a small linear subspace (capturing dominant shared variation), then run **novelty LOF** in that subspace—one explicit subspace plus local density, as a lightweight proxy for “find weirdness after dimensionality reduction.”

In [ ]:
title = 'sklearn substitute for Subspace Outlier Detection (SOD) (TruncatedSVD + novelty LOF)'
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline

n_comp = max(2, min(8, df_data_train.shape[1] - 1))
nn_sod = max(2, min(LOF_N, df_data_train.shape[0] - 1))
pipe = Pipeline(
    [
        ('svd', TruncatedSVD(n_components=n_comp, random_state=random_state)),
        (
            'lof',
            LocalOutlierFactor(n_neighbors=nn_sod, novelty=True, contamination=outliers_fraction),
        ),
    ]
)
clf = SklearnNegLabelsToPyOD(pipe)
run_detector(title, clf)


### sklearn substitute for Variational Auto Encoder

**Original idea (PyOD):** A VAE learns a latent distribution and reconstructs inputs, so large reconstruction error or low ELBO-like scores flag anomalies. **This cell:** we use **PCA reconstruction mean squared error**: a **linear** reconstruction model with a contamination-based threshold on error. It keeps the “reconstruction mismatch” intuition without PyTorch, at the cost of nonlinear latent structure.

In [ ]:
title = 'sklearn substitute for Variational Auto Encoder (PCA reconstruction error)'
n_pca_v = max(2, min(12, window_size - 1))
clf = PCAMSEReconstructionOutlier(contamination, n_components=n_pca_v, random_state=random_state)
run_detector(title, clf)


### sklearn substitute for Auto Encoder

**Original idea (PyOD):** A deep autoencoder compresses and expands data through hidden layers to learn nonlinear manifolds of “normal” windows. **This cell:** we train a **small MLPRegressor** to predict each window from itself (identity target), then threshold **per-window reconstruction MSE** at the desired contamination—nonlinear like a tiny autoencoder, but entirely within scikit-learn and WASM-friendly.

In [ ]:
title = 'sklearn substitute for Auto Encoder (MLP identity reconstruction)'
clf = MLPIdentityReconstructionOutlier(contamination, max_iter=120, random_state=random_state)
run_detector(title, clf)


## Next steps

Tune `window_size` and `contamination`, try your own series, or export scores with `clf.decision_function(df_data_test)` once a model fits.